# BirdCLEF+ 2025 - Pretrained models: seed repeats of the original A2 and A3 MLP heads

CPU only, no submission, no heads saved. We retrain the original unadapted heads (coral_heads control recipe) with seeds 42, 43 and 44.
**The seed changes head initialisation, batch order and dropout only; the stratified 80/20 split is seed 42 for every run.**
We compare seed 42 with the saved first-5s AUCs (0.9916 Perch, 0.9749 Bird-MAE). Std is the sample std (ddof 1) over the three seeds.

Output: `seed_repeats.json`.

In [ ]:
import os
N_THREADS = int(os.environ.get("M3_THREADS", os.cpu_count() or 4))
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = str(N_THREADS)         # set before numpy loads; all cores for the matrix work

import ast, glob, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

T0 = time.time()
torch.set_num_threads(N_THREADS)
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # local test only, never set on Kaggle
OUT = Path(os.environ.get("M3_OUT_DIR", "/kaggle/working"))
print("device: cpu | torch threads:", torch.get_num_threads(), "| os.cpu_count():", os.cpu_count())

CFG = dict(val_frac=0.2, seed=42, secondary_weight=0.5, batch_size=1024, weight_decay=1e-2,
           epochs=2 if SMOKE_TEST else 30, ridge=1.0, chunk=8192)
ENCODERS = dict(
    perch=dict(dim=1536, emb="perch_embeddings.npy", windows="perch_windows.csv",
               hidden=512, dropout=0.3, max_lr=1e-3, expected_first=0.9916),
    birdmae=dict(dim=768, emb="birdmae_embeddings.npy", windows="birdmae_windows.csv",
                 hidden=512, dropout=0.2, max_lr=3e-3, expected_first=0.9749))
SEEDS = (42, 43, 44)                                              # 42 first: the saved-value check and the timing guard use it
BUDGET_S = float(os.environ.get("M3_BUDGET_S", 45 * 60))          # env override: local guard test only
SAFETY = 1.25

def find_dir(marker, env):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{marker} not found under /kaggle/input")
    return Path(hits[0]).parent

DATA = find_dir("birdclef-2025/sample_submission.csv", "M3_DATA_DIR")
MELS = find_dir("train_index.csv", "M3_MELS_DIR")
PERCH_DIR = find_dir("perch_embeddings.npy", "M3_PERCH_DIR")
BIRDMAE_DIR = find_dir("birdmae_embeddings.npy", "M3_BIRDMAE_DIR")
DIRS = dict(perch=PERCH_DIR, birdmae=BIRDMAE_DIR)
print("Data:", DATA, "| M1 index:", MELS, "| perch:", PERCH_DIR, "| birdmae:", BIRDMAE_DIR)

## Split and labels (same table, call and seed as the original notebooks)

In [ ]:
index = pd.read_csv(MELS / "train_index.csv")      # same table, in the same order, that M1 split
species = list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:])
sp2idx = {s: i for i, s in enumerate(species)}
y_idx = index["primary_label"].astype(str).map(sp2idx)
assert y_idx.notna().all(), "primary_label not in submission columns"
y_idx = y_idx.astype(int).values

def parse_secondary(s):
    try:
        labs = ast.literal_eval(s) if isinstance(s, str) else []
    except (ValueError, SyntaxError):
        labs = []
    return [sp2idx[l] for l in labs if l in sp2idx]

sec_idx = [parse_secondary(s) for s in index["secondary_labels"]]

idx_tr, idx_va = train_test_split(
    np.arange(len(index)), test_size=CFG["val_frac"], stratify=y_idx, random_state=CFG["seed"])
assert not set(idx_tr) & set(idx_va)
print(len(index), "clips,", len(species), "species | train", len(idx_tr), "| val", len(idx_va), flush=True)

def macro_auc(y_true, y_pred):
    keep = y_true.sum(axis=0) > 0          # like the competition metric
    return roc_auc_score(y_true[:, keep], y_pred[:, keep], average="macro"), int(keep.sum())

def head_predict(head, emb):
    """Sigmoid probabilities from saved head arrays. Keep identical to the inference notebooks."""
    x = (emb - head["mean"]) / head["std"]
    n_layers = int(head["n_layers"])
    for i in range(n_layers):
        x = x @ head[f"W{i}"].T + head[f"b{i}"]
        if i < n_layers - 1:
            x = np.maximum(x, 0.0)       # ReLU
    return 1.0 / (1.0 + np.exp(-x))

## Head training (control recipe of coral_heads)

In [ ]:
def make_head(cfg):
    return nn.Sequential(nn.Linear(cfg["dim"], cfg["hidden"]), nn.ReLU(), nn.Dropout(cfg["dropout"]), nn.Linear(cfg["hidden"], len(species)))

def train_run(enc, cfg, d, seed, X):
    """One full control run. The seed sets head initialisation, batch order and dropout; the split stays seed 42."""
    t_start = time.time()
    torch.manual_seed(seed); np.random.seed(seed)
    mean, std = X[d.in_train].mean(axis=0), X[d.in_train].std(axis=0) + 1e-6
    Z = torch.from_numpy((X - mean) / std)
    tr_rows = torch.from_numpy(np.where(d.in_train)[0])

    @torch.no_grad()
    def predict(model, rows):
        model.eval(); out = []
        for i in range(0, len(rows), 4096):
            out.append(torch.sigmoid(model(Z[torch.from_numpy(rows[i:i + 4096])])).numpy())
        return np.concatenate(out)

    def clip_max_auc(model):
        p = pd.DataFrame(predict(model, d.rows_all_va)).groupby(d.va_groups).max().loc[d.order_clips].values
        return macro_auc(d.Y_clip, p)[0]

    model = make_head(cfg)
    epochs = CFG["epochs"]
    steps = int(np.ceil(len(tr_rows) / CFG["batch_size"]))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["max_lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg["max_lr"], total_steps=epochs * steps, pct_start=0.1)
    best = dict(first=-1.0); name = f"{enc}/seed{seed}"
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); total = 0.0
        perm = tr_rows[torch.randperm(len(tr_rows))]
        for s in range(steps):
            b = perm[s * CFG["batch_size"]:(s + 1) * CFG["batch_size"]]
            bce = nn.functional.binary_cross_entropy_with_logits(model(Z[b]), d.Y_t[b], reduction="none")
            loss = (bce.mean(1) * d.W_t[b]).sum() / d.W_t[b].sum()
            opt.zero_grad(set_to_none=True)
            loss.backward(); opt.step(); sched.step()
            total += loss.item()
        first, n_cls = macro_auc(d.Y_va, predict(model, d.rows_va))
        flag = ""
        if first > best["first"]:
            best = dict(first=first, clip_max=clip_max_auc(model), best_epoch=ep, n_classes=n_cls,
                        state={k: v.clone() for k, v in model.state_dict().items()}); flag = "  <- best"
        print(f"[{name}] ep {ep:2d}/{epochs} | loss {total / steps:.4f} | first-5s {first:.4f} | {time.time() - t0:.1f}s{flag}", flush=True)
    model.load_state_dict(best["state"]); model.eval()

    return dict(val_auc_first5s=float(best["first"]), val_auc_clip_max=float(best["clip_max"]), best_epoch=int(best["best_epoch"]),
                val_classes=int(best["n_classes"]), seed=seed, seconds=time.time() - t_start)

## Checks

In [ ]:
def check_control(enc, got, expected):
    diff = abs(got - expected)
    print(f"[{enc}] control first-5s {got:.4f} vs saved {expected:.4f} | diff {diff:.4f}", flush=True)
    if diff > 0.015:
        raise RuntimeError(f"{enc}: control differs from the saved value by {diff:.4f} (> 0.015), the recipe copy is wrong")
    if diff > 0.005:
        print(f"WARNING {enc}: control differs from the saved value by {diff:.4f} (> 0.005)", flush=True)
    return diff

def project_seconds(elapsed, prep_s, run_s, runs_left_here, encoders_after):
    """Elapsed plus the remaining work; a later encoder is costed like the current one (Perch is first and larger, so an upper bound)."""
    return elapsed + SAFETY * (runs_left_here * run_s + encoders_after * (prep_s + len(SEEDS) * run_s))

def check_budget(projected, limit=BUDGET_S):
    print(f"TIMING guard | projected notebook total {projected / 60:.1f} min (limit {limit / 60:.0f})", flush=True)
    if projected > limit:
        raise RuntimeError(f"Projected {projected / 60:.1f} min exceeds the {limit / 60:.0f} min limit, stopping")

def to_py(o):
    if isinstance(o, dict):
        return {k: to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_py(v) for v in o]
    if isinstance(o, np.generic):
        return o.item()
    return o

## Runs

In [ ]:
class Data:
    pass

def prepare(enc, cfg):
    t0 = time.time(); d = Data()
    cache = DIRS[enc]
    val_clips = pd.read_csv(cache / "val_clips.csv")
    if not np.array_equal(val_clips["clip"].values, idx_va):
        raise RuntimeError(f"{enc}: val_clips.csv does not match the recomputed split")
    if "filename" in index.columns and not (index["filename"].astype(str).values[idx_va] == val_clips["filename"].astype(str).values).all():
        raise RuntimeError(f"{enc}: val_clips.csv filenames do not match train_index.csv")
    E = np.load(cache / cfg["emb"]).astype(np.float32)
    windows = pd.read_csv(cache / cfg["windows"])
    if E.shape[1] != cfg["dim"] or len(E) != len(windows) or not np.isfinite(E).all():
        raise RuntimeError(f"{enc}: bad source embeddings {E.shape} vs {len(windows)} windows")
    d.E = E
    clip_of, win_of = windows["clip"].values, windows["win"].values
    d.in_train = np.isin(clip_of, idx_tr); in_val = np.isin(clip_of, idx_va)
    assert not (d.in_train & in_val).any() and (d.in_train | in_val).all()
    Y = np.zeros((len(windows), len(species)), np.float32)
    for row, c in enumerate(clip_of):
        Y[row, sec_idx[c]] = CFG["secondary_weight"]
        Y[row, y_idx[c]] = 1.0
    clip_windows = pd.Series(clip_of).map(pd.Series(clip_of).value_counts()).values
    d.W_t = torch.from_numpy((1.0 / clip_windows).astype(np.float32)); d.Y_t = torch.from_numpy(Y)
    d.rows_va = np.where(in_val & (win_of == 0))[0]
    d.Y_va = np.zeros((len(d.rows_va), len(species)), np.float32)
    d.Y_va[np.arange(len(d.rows_va)), y_idx[clip_of[d.rows_va]]] = 1.0
    d.rows_all_va = np.where(in_val)[0]
    d.va_groups = clip_of[d.rows_all_va]
    d.order_clips = np.unique(d.va_groups)
    d.Y_clip = np.zeros((len(d.order_clips), len(species)), np.float32)
    d.Y_clip[np.arange(len(d.order_clips)), y_idx[d.order_clips]] = 1.0
    print(f"[{enc}] windows: source {len(E)} | train {d.in_train.sum()} | val {in_val.sum()} | val first-5s {len(d.rows_va)}", flush=True)
    d.prep_s = time.time() - t0
    return d

results = dict(split="fixed: stratified 80/20 with seed 42 for every run; the seed varies head initialisation, batch order and dropout only",
               seeds=list(SEEDS), std_ddof=1, encoders={})
ENC_NAMES = list(ENCODERS)
for pos, enc in enumerate(ENC_NAMES):
    cfg = ENCODERS[enc]
    d = prepare(enc, cfg)
    entry = dict(n_source_windows=int(len(d.E)), n_train_windows=int(d.in_train.sum()), prep_seconds=d.prep_s, seeds={})
    for s_pos, seed in enumerate(SEEDS):
        r = train_run(enc, cfg, d, seed, d.E)
        entry["seeds"][str(seed)] = r
        print(f"RESULT [{enc}/seed{seed}] first-5s {r['val_auc_first5s']:.4f} | clip-max {r['val_auc_clip_max']:.4f} | best ep {r['best_epoch']} | {r['seconds']:.0f}s", flush=True)
        if seed == 42:
            entry["seed42_vs_saved"] = dict(saved_first5s=cfg["expected_first"], seed42_first5s=r["val_auc_first5s"],
                                            abs_diff=abs(r["val_auc_first5s"] - cfg["expected_first"]))
            if SMOKE_TEST:
                print("SMOKE_TEST=1: saved-value check skipped (synthetic data)")
            else:
                check_control(enc, r["val_auc_first5s"], cfg["expected_first"])
            check_budget(project_seconds(time.time() - T0, d.prep_s, r["seconds"], len(SEEDS) - 1 - s_pos, len(ENC_NAMES) - pos - 1))
        results["encoders"][enc] = entry
        json.dump(to_py(results), open(OUT / "seed_repeats.json", "w"), indent=1)
    firsts = np.array([v["val_auc_first5s"] for v in entry["seeds"].values()])
    clips = np.array([v["val_auc_clip_max"] for v in entry["seeds"].values()])
    entry.update(first5s_mean=float(firsts.mean()), first5s_std=float(firsts.std(ddof=1)),
                 clip_max_mean=float(clips.mean()), clip_max_std=float(clips.std(ddof=1)))
    results["encoders"][enc] = entry
    json.dump(to_py(results), open(OUT / "seed_repeats.json", "w"), indent=1)
    del d

## Summary

In [ ]:
results["total_seconds"] = time.time() - T0
json.dump(to_py(results), open(OUT / "seed_repeats.json", "w"), indent=1)
for enc, e in results["encoders"].items():
    runs = " | ".join(f"seed {s}: {v['val_auc_first5s']:.4f}/{v['val_auc_clip_max']:.4f}/ep{v['best_epoch']}" for s, v in e["seeds"].items())
    print(f"{enc:8s} | {runs} | first-5s mean {e['first5s_mean']:.4f} std {e['first5s_std']:.4f} | clip-max mean {e['clip_max_mean']:.4f} std {e['clip_max_std']:.4f}")
parts = " | ".join(f"{enc} prep {e['prep_seconds']:.0f}s + runs " + "/".join(f"{v['seconds']:.0f}" for v in e["seeds"].values()) + "s" for enc, e in results["encoders"].items())
print(f"TIMING | {parts} | notebook total {results['total_seconds'] / 60:.1f} min")